## Conecta ao banco de dados

In [1]:
import psycopg2

conn = psycopg2.connect(
    dbname="ibge_origem",
    user="admin",
    password="admin",
    host="localhost"
)

## Gera os csvs das tabelas

In [2]:
import csv
import io

cur = conn.cursor()

cur.execute("""
    SELECT tablename
    FROM pg_tables
    WHERE schemaname = 'public';
""")

csv_files = {}

tables = cur.fetchall()

for (table,) in tables:
    buffer = io.StringIO()
    cur.execute(f'SELECT * FROM "{table}"')
    writer = csv.writer(buffer)
    writer.writerow([desc[0] for desc in cur.description])
    writer.writerows(cur.fetchall())
    csv_files[f"{table}.csv"] = buffer.getvalue()
    buffer.close()

cur.close()
conn.close()

## Conecta ao Servidor MinIo

In [3]:
import boto3
from botocore.config import Config

MINIO_ENDPOINT = "http://127.0.0.1:9000"
MINIO_ACCESS_KEY = "minioadmin"
MINIO_SECRET_KEY = "miniopass"

s3_client = boto3.client(
    "s3",
    endpoint_url=MINIO_ENDPOINT,
    aws_access_key_id=MINIO_ACCESS_KEY,
    aws_secret_access_key=MINIO_SECRET_KEY,
    region_name="us-east-1",
    config=Config(s3={"addressing_style": "path"}),
)

## Envia os arquivos para landing

In [4]:
bucket_name = "landing"

for filename, content in csv_files.items():

    s3_client.put_object(
        Bucket=bucket_name,
        Key=f"{filename}",
        Body=content.encode("utf-8"),
        ContentType="text/csv"
    )

    print(f"Uploaded: {filename}")

Uploaded: inpc_valor.csv
Uploaded: pib_municipio.csv
Uploaded: pnad_domicilio.csv
Uploaded: pnad_morador.csv
Uploaded: pnad_ocupacao.csv
Uploaded: populacao_municipio.csv
Uploaded: producao_origem_animal.csv
Uploaded: rebanho_municipio.csv
Uploaded: ipca_valor.csv
Uploaded: municipio.csv
Uploaded: producao_agricola.csv
Uploaded: categoria_produto.csv
